# Heart Disease Classifier - Inference Demo

This notebook demonstrates how to:
1. Load the saved model and make predictions locally
2. Send requests to the running FastAPI endpoint

In [5]:
import sys
sys.path.insert(0, '..')

import joblib
import pandas as pd
import requests
import json

## 1. Local Inference with Saved Model

In [6]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

model = joblib.load('../model/heart_disease_model.joblib')

clf = model.named_steps.get("classifier") or model.steps[-1][1]
if hasattr(clf, "predict") and not hasattr(clf, "multi_class"):
    clf.multi_class = "auto"

print(f'Model type: {type(model).__name__}')
print(f'Pipeline steps: {[step[0] for step in model.steps]}')

Model type: Pipeline
Pipeline steps: ['preprocessor', 'classifier']


In [7]:
sample_patients = pd.DataFrame([
    {'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145.0, 'chol': 233.0, 'fbs': 1,
     'restecg': 0, 'thalach': 150.0, 'exang': 0, 'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 6},
    {'age': 45, 'sex': 0, 'cp': 1, 'trestbps': 120.0, 'chol': 200.0, 'fbs': 0,
     'restecg': 0, 'thalach': 170.0, 'exang': 0, 'oldpeak': 0.0, 'slope': 1, 'ca': 0, 'thal': 3},
    {'age': 67, 'sex': 1, 'cp': 4, 'trestbps': 160.0, 'chol': 286.0, 'fbs': 0,
     'restecg': 2, 'thalach': 108.0, 'exang': 1, 'oldpeak': 1.5, 'slope': 2, 'ca': 3, 'thal': 3},
])

predictions = model.predict(sample_patients)
probabilities = model.predict_proba(sample_patients)

for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    label = 'Disease' if pred == 1 else 'No Disease'
    print(f'Patient {i+1}: {label} (confidence: {prob[pred]:.4f}, disease probability: {prob[1]:.4f})')

Patient 1: No Disease (confidence: 0.8489, disease probability: 0.1511)
Patient 2: No Disease (confidence: 0.9926, disease probability: 0.0074)
Patient 3: Disease (confidence: 0.9833, disease probability: 0.9833)


## 2. API Inference (requires running server)

Start the server first:
```bash
cd /path/to/MLOPS_Assignment
uvicorn src.serving.app:app --port 8080
```

In [8]:
API_URL = 'http://localhost:8080'

try:
    health = requests.get(f'{API_URL}/health', timeout=5)
    print(f'Health check: {health.json()}')
except requests.ConnectionError:
    print('API server is not running. Start it with: uvicorn src.serving.app:app --port 8080')

Health check: {'status': 'healthy', 'model_loaded': True}


In [9]:
sample = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145.0, 'chol': 233.0, 'fbs': 1,
    'restecg': 0, 'thalach': 150.0, 'exang': 0, 'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 6
}

try:
    resp = requests.post(f'{API_URL}/predict', json=sample, timeout=5)
    result = resp.json()
    print(f'Request:  {json.dumps(sample)}')
    print(f'Response: {json.dumps(result, indent=2)}')
except requests.ConnectionError:
    print('API server is not running.')

Request:  {"age": 63, "sex": 1, "cp": 3, "trestbps": 145.0, "chol": 233.0, "fbs": 1, "restecg": 0, "thalach": 150.0, "exang": 0, "oldpeak": 2.3, "slope": 0, "ca": 0, "thal": 6}
Response: {
  "prediction": 0,
  "confidence": 0.8745,
  "disease_probability": 0.1255
}
